YouTube AAC & OPUS audio combiner to WAV (44kHz/32-bit float) by IncT |
[Info](https://discord.com/channels/708579735583588363/773763762887852072/1492197858005356545)


In [ ]:
#@markdown ##Installation & GDrive mounting
from google.colab import drive
drive.mount('/content/drive')
!pip install yt-dlp
!pip install scipy
!pip install ffmpeg
!git clone https://github.com/deton24/yt_audio_combine_by_IncT

In [ ]:
# @title Execute
# @markdown Paste URL to YT video below:
%cd '/content/'

url_address = "https://www.youtube.com/watch?v=XXXXXXXXXX" # @param {type:"string"}

!python /content/yt_audio_combine_by_IncT/yt_audio_combine.py "$url_address"

In [ ]:
#@markdown ###Copy the results to "GDrive/YT" folder <br>
#@markdown ######(delay of a few seconds might occur on your GDrive)
!cp -r '/content/combined/yt' '/content/drive/MyDrive'

In [ ]:

#@markdown <center><h3><b>Pack it</b></h3><br>

from IPython.display import clear_output

!sudo curl -L https://www.7-zip.org/a/7z2301-linux-x64.tar.xz -o /usr/local/bin/7zip.tar.xz
%cd /usr/local/bin/
!7z e -y /usr/local/bin/7zip.tar.xz
!7z x -y /usr/local/bin/7zip.tar 7zz
!rm /usr/local/bin/7zip.tar.xz
!rm /usr/local/bin/7zip.tar
!sudo chmod +x /usr/local/bin/7zz
clear_output()
!7zz | sed -n 2p

import os

sourcePath = "/content/combined/yt/" #@param {type:"string"}
compressType = "7z" #@param ["zip", "7z", "tar", "gzip", "bzip2", "xz", "wim"]
Password = "" #@param {type:"string"}
Split = "no" #@param ["no", "10m", "100m", "500m", "1g", "2g"] {allow-input: true}
compressLevel = 7 #@param {type:"slider", min:0, max:9, step:1}
saveToSourceLocation = True #@param {type:"boolean"}

commandLine = "-t" + compressType + " -mx=" + str(compressLevel)

if len(Password) > 0:
    commandLine = commandLine + " -p" + '"' + Password + '"'

if Split != "no":
    commandLine = commandLine + " -v" + '"' + Split + '"'

if os.path.isfile(sourcePath):
    sourceName = os.path.splitext(os.path.basename(os.path.abspath(sourcePath)))[0]
    sourceFolder = os.path.dirname(os.path.abspath(sourcePath))
else:
    sourceName = os.path.split(os.path.abspath(sourcePath))[1]
    sourceFolder = os.path.split(os.path.abspath(sourcePath))[0]

if saveToSourceLocation:
    compressPath = os.path.join(sourceFolder, "compressed")
    commandLine = commandLine + ' "' + compressPath + '"'
else:
    outputPath = input("outputPath: ")
    if outputPath.endswith('.zip') or outputPath.endswith('.7z') or outputPath.endswith('.tar') or outputPath.endswith('.gz') or outputPath.endswith('.bz2') or outputPath.endswith('.xz') or outputPath.endswith('.wim'):
        sourceName = os.path.splitext(os.path.basename(os.path.abspath(outputPath)))[0]
        sourceFolder = os.path.dirname(os.path.abspath(outputPath))
    else:
        if not os.path.exists(outputPath):
            os.makedirs(outputPath)
        sourceFolder = outputPath
    compressPath = os.path.join(sourceFolder, "compressed")
    commandLine = commandLine + ' "' + compressPath + '"'

if compressType == "gzip":
    formatFile = "gz"
elif compressType == "bzip2":
    formatFile = "bz2"
else:
    formatFile = compressType

!7zz a {commandLine} "{sourcePath}"
saveFile = os.path.join(sourceFolder, sourceName)
compressPath = compressPath + '.' + formatFile
saveFile = saveFile + '.' + formatFile
os.rename(compressPath, saveFile)

In [ ]:
#@markdown ###Copy all .7z files to 'GDrive\yt'

import glob, os, shutil

files = glob.iglob(os.path.join('/content/combined/', "*.7z"))
for file in files:
    if os.path.isfile(file):
        shutil.copy2(file, '/content/drive/MyDrive/yt/')

In [ ]:
#@markdown ###### (optional) Convert from 32-bit float to 24-bit and copy the converted file to "GDrive\YT\conv"
# It doesn't normalize (clipping here can't be turned down)

file_to_convert = "/content/combined/yt/Your video title [video ID].wav" # @param {type:"string"}
output_file = "/content/drive/MyDrive/yt/conv/conv24.wav" # @param {type:"string"}

# Utworzenie folderu, jeśli nie istnieje
import os
newpath = r'/content/drive/MyDrive/yt/conv/'
if not os.path.exists(newpath):
    os.makedirs(newpath)

# Pobieranie samej ścieżki do folderu z podanego pliku wyjściowego
destination_folder = os.path.dirname(output_file)

# Tworzenie folderu docelowego, jeśli nie istnieje
if destination_folder and not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# Uruchomienie ffmpeg z wykorzystaniem zmiennych
!ffmpeg -i "$file_to_convert" -c:a pcm_s24le "$output_file"